<a href="https://colab.research.google.com/github/Arfa-Tariq/learning-ai-engineering/blob/main/projects/04-building-systems/02-Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Evaluate Inputs: Classification
 - In this section, we'll focus on tasks to evaluate inputs, which can be important for ensuring the quality and safety of the system.

In [1]:
!pip install -q groq

from groq import Groq
from google.colab import userdata
import tiktoken

GROQ_API_KEY = userdata.get('groq_key')
client = Groq(api_key=GROQ_API_KEY)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 13.9 MB/s eta 0:00:00


In [2]:
def get_completion_from_messages(messages,
                                 model="llama-3.3-70b-versatile",
                                 temperature=0,
                                 max_tokens=8000):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, # this is the degree of randomness of the model's output
        max_tokens=max_tokens, # the maximum number of tokens the model can ouptut
    )
    return response.choices[0].message.content

- For tasks in which lots of independent sets
of instructions are needed to handle different cases,
it can be beneficial to first classify the type of query,
and then use that classification to determine which instructions to use.
- This can be achieved by defining fixed categories and hard-coding
instructions that are relevant for handling tasks in
a given category.

---

For instance, when building a customer service assistant,
it might be important to first classify the type of query,
and then determine which instructions to use based on
that classification

In [3]:
delimiter = "####"
system_message = f"""
You will be provided with customer service queries. \
The customer service query will be delimited with \
{delimiter} characters.
Classify each query into a primary category \
and a secondary category.
Provide your output in json format with the \
keys: primary and secondary.

Primary categories: Billing, Technical Support, \
Account Management, or General Inquiry.

Billing secondary categories:
Unsubscribe or upgrade
Add a payment method
Explanation for charge
Dispute a charge

Technical Support secondary categories:
General troubleshooting
Device compatibility
Software updates

Account Management secondary categories:
Password reset
Update personal information
Close account
Account security

General Inquiry secondary categories:
Product information
Pricing
Feedback
Speak to a human

"""
user_message = f"""\
I want you to delete my profile and all of my user data"""
messages =  [
{'role':'system',
 'content': system_message},
{'role':'user',
 'content': f"{delimiter}{user_message}{delimiter}"},
]
response = get_completion_from_messages(messages)
print(response)

```json
{
  "primary": "Account Management",
  "secondary": "Close account"
}
```


In [4]:
user_message = f"""\
Tell me more about your flat screen tvs"""
messages =  [
{'role':'system',
 'content': system_message},
{'role':'user',
 'content': f"{delimiter}{user_message}{delimiter}"},
]
response = get_completion_from_messages(messages)
print(response)

```json
{
  "primary": "General Inquiry",
  "secondary": "Product information"
}
```


In [5]:
user_message = f"""\
Can you please explain these 20$ worth of extra charges in my subscription"""
messages =  [
{'role':'system',
 'content': system_message},
{'role':'user',
 'content': f"{delimiter}{user_message}{delimiter}"},
]
response = get_completion_from_messages(messages)
print(response)

```json
{
  "primary": "Billing",
  "secondary": "Explanation for charge"
}
```
